# Optimal Control with OpEn

This notebook is based on the OpEn getting-started optimal control example:

- https://alphaville.github.io/optimization-engine/docs/python-ocp-1

We define a nonlinear optimal control problem with:

- two state variables
- one control input
- a stage cost and a terminal cost
- hard input constraints
- a path constraint handled with ALM

The build step generates a Python-callable optimizer inside the notebook environment, so it can take a little while the first time you run it.

In [ ]:
import casadi.casadi as cs
import matplotlib.pyplot as plt
import numpy as np
import opengen as og

In [ ]:
optimizer_name = "ocp_alm_demo"

ocp = og.ocp.OptimalControlProblem(nx=2, nu=1, horizon=20)

ocp.add_parameter("x0", 2)
ocp.add_parameter("xref", 2, default=[0.0, 0.0])
ocp.add_parameter("q", 1, default=1)
ocp.add_parameter("r", 1, default=0.1)
ocp.add_parameter("a", 1, default=0.8)
ocp.add_parameter("xmin", 1, default=-1)

ocp.with_dynamics(
    lambda x, u, param: cs.vertcat(
        0.98 * cs.sin(x[0]) + x[1],
        0.1 * x[0] ** 2 - 0.5 * x[0] + param["a"] * x[1] + u[0],
    )
)

ocp.with_stage_cost(
    lambda x, u, param, _t: param["q"] * cs.dot(x - param["xref"], x - param["xref"])
    + param["r"] * cs.dot(u, u)
)

ocp.with_terminal_cost(
    lambda x, param: 100 * cs.dot(x - param["xref"], x - param["xref"])
)

ocp.with_path_constraint(
    lambda x, u, param, _t: x[1] - param["xmin"],
    kind="alm",
    set_c=og.constraints.Rectangle([0.0], [np.inf]),
)

ocp.with_input_constraints(og.constraints.BallInf(radius=0.2))
ocp

In [ ]:
ocp_optimizer = og.ocp.OCPBuilder(
    ocp,
    metadata=og.config.OptimizerMeta().with_optimizer_name(optimizer_name),
    build_configuration=og.config.BuildConfiguration()
    .with_build_python_bindings()
    .with_rebuild(True),
    solver_configuration=og.config.SolverConfiguration()
    .with_tolerance(1e-5)
    .with_delta_tolerance(1e-5)
    .with_preconditioning(True)
    .with_penalty_weight_update_factor(1.8)
    .with_max_inner_iterations(2000)
    .with_max_outer_iterations(40),
).build()

In [ ]:
result = ocp_optimizer.solve(x0=[0.4, 0.2], q=30, r=1, xmin=-0.2)
result

In [ ]:
print("exit_status   =", result.exit_status)
print("solve_time_ms =", result.solve_time_ms)
print("num_inputs    =", len(result.inputs))
print("num_states    =", len(result.states))
print("first input   =", result.inputs[0])
print("last state    =", result.states[-1])

In [ ]:
u_values = np.array(result.inputs, dtype=float).reshape(-1)
x_values = np.array(result.states, dtype=float)
t_inputs = np.arange(len(u_values))
t_states = np.arange(len(x_values))

In [ ]:
plt.figure(figsize=(8, 4))
plt.step(t_inputs, u_values, where="post", linewidth=2)
plt.title("Optimal input sequence")
plt.xlabel("Time step")
plt.ylabel("u")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(t_states, x_values[:, 0], marker="o", linewidth=2, label="$x_1$")
plt.plot(t_states, x_values[:, 1], marker="s", linewidth=2, label="$x_2$")
plt.title("State trajectory")
plt.xlabel("Time step")
plt.ylabel("state")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()